# 01 Data Check — PWR Abnormality Dataset

这个 notebook 只做一件事：**把原始数据看清楚**。

目标：
1. 读取原始数据
2. 检查列名、数据类型、缺失值、重复值
3. 找出 `Flow2` 的格式问题并修复
4. 确认 `Readings` 是编号列，不参与建模
5. 给出数据检查结论，为后续 baseline 建模做准备


## 0. 导入库

这里只做数据检查，所以只用到 `pandas`。

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)
pd.set_option('display.max_rows', 100)


## 1. 读取原始数据

这里先直接读取 csv，不做任何处理。

In [2]:
data_path = '../data/PWR Abnormality Dataset.csv'
df_raw = pd.read_csv(data_path)

print('原始数据形状：', df_raw.shape)
df_raw.head()

原始数据形状： (12267, 17)


,Readings,Temperature,Pressure,Flow1,Flow2,VRR12,VRR22,VRR23,VRR33,VRS01,VRS03,VRS21,VRS31,VRS02,VRI01,VRI02,VRI03
0,1,248.852987,9.689813,4462.130014,13302.9265,19.060938,0.059119,0.050589,0.111864,0.033951,0.047812,0.232627,0.253775,0.400726,1.763223,0.003031,0.004995
1,2,269.315740,1.279532,4480.252595,13784.45225,19.062128,0.059089,0.048788,0.111340,0.034060,0.052611,0.233342,0.315067,0.128517,1.769272,0.003164,0.004999
2,3,94.320644,6.280686,4325.270376,12899.98773,19.061641,0.058145,0.048552,0.111118,0.033859,0.053999,0.233387,0.380191,0.321816,1.768585,0.003321,0.004996
3,4,271.019823,0.669886,4481.761795,13733.0061,19.062453,0.058236,0.049514,0.112658,0.034049,0.050767,0.233850,0.329846,0.132092,1.772432,0.003061,0.004998
4,5,207.876262,6.806043,4425.839490,13500.22222,19.061101,0.058378,0.049809,0.111252,0.033996,0.052669,0.233619,0.390729,0.247973,1.772356,0.003060,0.004994


## 2. 基础信息检查

先看列名、数据类型、基础统计。

In [3]:
print('列名：')
print(df_raw.columns)

print('\n数据类型：')
print(df_raw.dtypes)

print('\n描述统计：')
display(df_raw.describe(include='all'))

列名：
Index(['Readings', 'Temperature', 'Pressure', 'Flow1', 'Flow2', 'VRR12', 'VRR22', 'VRR23', 'VRR33', 'VRS01', 'VRS03', 'VRS21', 'VRS31', 'VRS02', 'VRI01', 'VRI02', 'VRI03'], dtype='object')

数据类型：
Readings         int64
Temperature    float64
Pressure       float64
Flow1          float64
Flow2           object
VRR12          float64
VRR22          float64
VRR23          float64
VRR33          float64
VRS01          float64
VRS03          float64
VRS21          float64
VRS31          float64
VRS02          float64
VRI01          float64
VRI02          float64
VRI03          float64
dtype: object

描述统计：


,Readings,Temperature,Pressure,Flow1,Flow2,VRR12,VRR22,VRR23,VRR33,VRS01,VRS03,VRS21,VRS31,VRS02,VRI01,VRI02,VRI03
count,12267.000000,12267.000000,12267.000000,12267.000000,12267,12267.000000,12267.000000,12267.000000,12267.000000,12267.000000,12267.000000,12267.000000,12267.000000,12267.000000,12267.000000,12267.000000,12267.000000
unique,NaN,NaN,NaN,NaN,12266,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,NaN,NaN,NaN,12230.55415,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,NaN,NaN,NaN,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
mean,6134.000000,178.606423,8.345774,4399.392038,NaN,18.651021,0.058502,0.050002,0.112421,0.033884,0.051051,0.232717,0.316198,0.236109,1.767099,0.003177,0.004997
std,3541.322211,65.110042,4.827976,57.263075,NaN,2.753532,0.000546,0.001253,0.000756,0.000130,0.002449,0.001353,0.045526,0.094024,0.003763,0.000107,0.000002
min,1.000000,65.787100,0.116647,4300.013982,NaN,0.190607,0.057566,0.047817,0.111088,0.033655,0.046912,0.230400,0.239994,0.079196,1.760482,0.003000,0.004990
25%,3067.500000,122.269470,4.117878,4349.994355,NaN,19.061180,0.058030,0.048914,0.111777,0.033772,0.048894,0.231535,0.276082,0.155658,1.763901,0.003085,0.004995
50%,6134.000000,179.014289,8.360574,4399.781503,NaN,19.061712,0.058486,0.050030,0.112431,0.033885,0.051040,0.232694,0.315689,0.234240,1.767064,0.003178,0.004997
75%,9200.500000,234.745227,12.549597,4448.686126,NaN,19.062237,0.058974,0.051087,0.113067,0.033996,0.053176,0.233896,0.356154,0.317188,1.770378,0.003271,0.004998


## 3. 缺失值与重复值检查

这一步是最基础的数据体检。

In [4]:
print('各列缺失值数量：')
print(df_raw.isnull().sum())

print('\n重复行数量：', df_raw.duplicated().sum())

各列缺失值数量：
Readings       0
Temperature    0
Pressure       0
Flow1          0
Flow2          0
VRR12          0
VRR22          0
VRR23          0
VRR33          0
VRS01          0
VRS03          0
VRS21          0
VRS31          0
VRS02          0
VRI01          0
VRI02          0
VRI03          0
dtype: int64

重复行数量： 0


## 4. 定位 `Flow2` 的格式问题

前面已经发现 `Flow2` 被读成了 `object`，这说明它不是纯数值列。

下面先检查：
- 前几行长什么样
- 这一列元素实际都是什么类型
- 有多少个值不能直接转成数值

In [5]:
print('Flow2 前10个值：')
print(df_raw['Flow2'].head(10))

print('\nFlow2 元素类型统计：')
print(df_raw['Flow2'].apply(type).value_counts())

flow2_numeric_try = pd.to_numeric(df_raw['Flow2'], errors='coerce')
print('\n尝试直接转数值后，缺失数量：', flow2_numeric_try.isna().sum())

print('\n无法直接转成数值的样本：')
display(df_raw.loc[flow2_numeric_try.isna(), ['Readings', 'Flow2']].head(20))

Flow2 前10个值：
0     13302.9265
1    13784.45225
2    12899.98773
3     13733.0061
4    13500.22222
5    12895.87696
6    13212.69614
7    13399.78489
8    12466.70757
9    12907.71095
Name: Flow2, dtype: object

Flow2 元素类型统计：
Flow2
<class 'str'>    12267
Name: count, dtype: int64

尝试直接转数值后，缺失数量： 1

无法直接转成数值的样本：


,Readings,Flow2
12003,12004,"13077,1"


## 5. 修复 `Flow2`

这里的核心问题是：有个别值用了逗号小数格式，比如 `13077,1`。

处理思路：
1. 先转成字符串
2. 去掉两边空格
3. 把逗号改成点
4. 再转成数值型

In [6]:
df = df_raw.copy()

df['Flow2'] = df['Flow2'].astype(str).str.strip().str.replace(',', '.', regex=False)
df['Flow2'] = pd.to_numeric(df['Flow2'], errors='coerce')

print('修复后 Flow2 的数据类型：', df['Flow2'].dtype)
print('修复后 Flow2 的缺失值数量：', df['Flow2'].isna().sum())
df[['Readings', 'Flow2']].head(10)

修复后 Flow2 的数据类型： float64
修复后 Flow2 的缺失值数量： 0


,Readings,Flow2
0,1,13302.92650
1,2,13784.45225
2,3,12899.98773
3,4,13733.00610
4,5,13500.22222
5,6,12895.87696
6,7,13212.69614
7,8,13399.78489
8,9,12466.70757
9,10,12907.71095


## 6. 确认 `Readings` 是否只是编号列

如果 `Readings` 只是 1,2,3,... 这样的序号，那么后续建模时应该删除。

In [7]:
print('Readings 前5个值：')
print(df['Readings'].head())

print('\nReadings 后5个值：')
print(df['Readings'].tail())

print('\nReadings 是否唯一：', df['Readings'].is_unique)


Readings 前5个值：
0    1
1    2
2    3
3    4
4    5
Name: Readings, dtype: int64

Readings 后5个值：
12262    12263
12263    12264
12264    12265
12265    12266
12266    12267
Name: Readings, dtype: int64

Readings 是否唯一： True


## 7. 生成建模用数据表（仅用于检查，不做建模）

这里先删掉 `Readings`，看删完之后的数据形状和列名。

In [8]:
df_model_check = df.drop(columns=['Readings'])

print('删掉 Readings 后的数据形状：', df_model_check.shape)
print('\n删掉 Readings 后的列名：')
print(df_model_check.columns)

print('\n删掉 Readings 后的数据类型：')
print(df_model_check.dtypes)

删掉 Readings 后的数据形状： (12267, 16)

删掉 Readings 后的列名：
Index(['Temperature', 'Pressure', 'Flow1', 'Flow2', 'VRR12', 'VRR22', 'VRR23', 'VRR33', 'VRS01', 'VRS03', 'VRS21', 'VRS31', 'VRS02', 'VRI01', 'VRI02', 'VRI03'], dtype='object')

删掉 Readings 后的数据类型：
Temperature    float64
Pressure       float64
Flow1          float64
Flow2          float64
VRR12          float64
VRR22          float64
VRR23          float64
VRR33          float64
VRS01          float64
VRS03          float64
VRS21          float64
VRS31          float64
VRS02          float64
VRI01          float64
VRI02          float64
VRI03          float64
dtype: object


## 8. 数据检查小结

### 当前结论
- 数据共 **12267 行，17 列**
- 原始数据 **无缺失值、无重复值**
- `Flow2` 存在格式问题（个别值使用逗号小数格式），已修复为数值型
- `Readings` 是编号列，后续建模时应删除
- 清洗后可用于建模的特征共有 **16 列**

### 下一步
下一本 notebook 将进入：
**02_baseline_isolation_forest.ipynb**

在那里会完成：
- 标准化
- Isolation Forest baseline
- 异常标签与异常分数生成
- 基础可视化